# Domain 7: Security and Safety
**Exam**: Claude Certified Developer – Foundations (CCDV-F)
**Weighting**: 8.1% of exam
**Depth**: deep

## Domain Overview

This domain covers AI application security (prompt injection, jailbreak defense,
untrusted input handling, PII, AuthN/AuthZ/confidentiality/integrity), guardrails and safe deployment
(content policy, guardrail layering, secure-by-design/least-privilege), Claude Hooks for guardrails,
and identity/secrets/key management.

**Why it matters**: developers ship the code that actually enforces (or fails to enforce) security and
safety controls in production.

**Key concepts to know**: prompt injection/jailbreak defense, layered guardrails, hooks for
deterministic safety controls, and secrets/key management.

In [ ]:
# Setup: a small helper so every cell in this notebook runs end-to-end,
# whether or not you have an ANTHROPIC_API_KEY configured.
import os

def call_claude(prompt, system=None, model="claude-sonnet-4-5", max_tokens=600):
    """Calls the real Claude API if ANTHROPIC_API_KEY is set; otherwise returns a
    simulated response so this notebook is runnable without API access."""
    try:
        import anthropic
        api_key = os.environ.get("ANTHROPIC_API_KEY")
        if not api_key:
            raise RuntimeError("no ANTHROPIC_API_KEY set")
        client = anthropic.Anthropic(api_key=api_key)
        kwargs = {"model": model, "max_tokens": max_tokens, "messages": [{"role": "user", "content": prompt}]}
        if system:
            kwargs["system"] = system
        resp = client.messages.create(**kwargs)
        return resp.content[0].text
    except Exception as e:
        return (f"[SIMULATED RESPONSE - set ANTHROPIC_API_KEY and `pip install anthropic` "
                f"to call the real API]\nPrompt received ({len(prompt)} chars): {prompt[:150]}...")

print("call_claude() ready. Set ANTHROPIC_API_KEY as an environment variable to hit the real API.")

## Key Concepts

### AI Application Security: prompt injection, jailbreaks, untrusted input, PII

Prompt injection uses untrusted content to smuggle instructions; jailbreaks try to bypass
model-level safety training directly. Defenses include isolating untrusted content (as in Domain 6),
least-privilege tool access so even a successful injection can't cause damage, and never logging/
persisting PII without justification and safeguards.

In [ ]:
def contains_pii(text: str) -> bool:
    import re
    return bool(re.search(r"[\w.]+@[\w.]+\.\w+", text) or re.search(r"\b\d{3}-\d{2}-\d{4}\b", text))

def redact_pii(text: str) -> str:
    import re
    text = re.sub(r"[\w.]+@[\w.]+\.\w+", "[email]", text)
    text = re.sub(r"\b\d{3}-\d{2}-\d{4}\b", "[ssn]", text)
    return text

sample = "Contact jane@example.com or SSN 123-45-6789 for details."
print(contains_pii(sample), redact_pii(sample))

### Guardrails and Safe Deployment: content policy, layering, secure-by-design

Secure-by-design means starting from least privilege and adding capabilities only as
justified, rather than starting broad and trying to restrict later. Guardrails should be layered
(input, model, output) as covered in earlier certifications' governance domains, applied here at the
implementation level.

In [ ]:
def least_privilege_tool_set(role: str) -> list[str]:
    role_tools = {"summarizer": ["read_document"], "refund_agent": ["get_customer", "process_refund"]}
    return role_tools.get(role, [])

print(least_privilege_tool_set("summarizer"))  # should NOT include process_refund or delete tools

### Claude Hooks for guardrails and safety controls

Hooks provide deterministic interception points (e.g., before/after a tool call) to enforce
safety rules programmatically, rather than relying solely on the model following instructions —
critical for preventing destructive actions like unauthorized deletes or over-limit transactions.

In [ ]:
def destructive_action_hook(tool_name: str, tool_input: dict) -> bool:
    """Returns True if the action is allowed to proceed."""
    if tool_name == "delete_database" :
        return False  # never allow via automated hook - always require manual out-of-band action
    if tool_name == "process_refund" and tool_input.get("amount", 0) > 1000:
        return False
    return True

print(destructive_action_hook("delete_database", {}))
print(destructive_action_hook("process_refund", {"amount": 200}))

### Identity, Secrets, and Key Management

Secrets (API keys, tokens) should never be hard-coded or committed; use environment
variable expansion or a secrets manager. Identity validation and authorization checks (does this
caller have permission for this action?) should happen server-side, not be trusted from client input.

In [ ]:
import os

def get_secret(name: str) -> str:
    value = os.environ.get(name)
    if not value:
        raise RuntimeError(f"Secret {name} not set - never hard-code secrets in source")
    return value

def is_authorized(user_role: str, action: str) -> bool:
    permissions = {"admin": {"delete", "refund", "read"}, "agent": {"read", "refund"}, "viewer": {"read"}}
    return action in permissions.get(user_role, set())

print(is_authorized("agent", "refund"), is_authorized("agent", "delete"))

## Mini Hands-On Project

**Covers**: all Domain 7 sub-objectives.

**Scenario**: Harden a refund-processing agent against prompt injection, over-privilege, and
unauthorized actions.

### Step 1: Screen and redact PII before logging

In [ ]:
ticket_text = "Customer email jane@example.com is requesting a refund."
print(redact_pii(ticket_text))

### Step 2: Apply least-privilege tools and a destructive-action hook

In [ ]:
print(least_privilege_tool_set("refund_agent"))
print(destructive_action_hook("process_refund", {"amount": 1500}))

### Step 3: Enforce authorization server-side

In [ ]:
print(is_authorized("agent", "refund"))
print(is_authorized("viewer", "refund"))

## Lab Exercises

### Exercise 1: PII redaction pipeline

**Objective**: Extend redact_pii to also handle phone numbers.

**Produce**: An updated redaction function tested on a mixed-PII string.

In [ ]:
def redact_pii_v2(text: str) -> str:
    text = redact_pii(text)
    # TODO: add a regex substitution for phone numbers like 555-123-4567
    return text

print(redact_pii_v2("Call 555-123-4567 or email jane@example.com, SSN 123-45-6789"))

# Expected output: All three PII types (email, SSN, phone) correctly redacted.

In [ ]:
# STRETCH GOAL: write a function that returns True only if ALL PII has been successfully redacted
# from the output (i.e., no PII patterns remain).
def fully_redacted(text: str) -> bool:
    pass
print(fully_redacted(redact_pii_v2("Call 555-123-4567")))

# Expected output: True, confirming no PII patterns remain after redaction.

### Exercise 2: Least-privilege audit for a new role

**Objective**: Design a least-privilege tool set for a new 'billing_viewer' role.

**Produce**: A tool list appropriate for a read-only billing viewer.

In [ ]:
# TODO: add a "billing_viewer" entry to a copy of the role_tools mapping used in least_privilege_tool_set,
# granting ONLY read-oriented billing tools
def least_privilege_tool_set_v2(role: str) -> list[str]:
    role_tools = {
        "summarizer": ["read_document"],
        "refund_agent": ["get_customer", "process_refund"],
        "billing_viewer": [],  # TODO
    }
    return role_tools.get(role, [])

print(least_privilege_tool_set_v2("billing_viewer"))

# Expected output: A tool list containing only read/view billing tools, no refund/write/delete capability.

In [ ]:
# STRETCH GOAL: write an assertion-based test that fails if any tool name containing
# "process", "delete", or "write" ever appears in the billing_viewer tool set.
def test_billing_viewer_is_read_only():
    tools = least_privilege_tool_set_v2("billing_viewer")
    assert not any(kw in t for t in tools for kw in ["process", "delete", "write"])
test_billing_viewer_is_read_only()
print("passed")

# Expected output: The test passes, confirming no write/delete/process tools leaked into the read-only role.

### Exercise 3: Hook-based enforcement for a new business rule

**Objective**: Extend destructive_action_hook to also block deletes without a 2-person approval flag.

**Produce**: An updated hook and 3 test cases.

In [ ]:
def destructive_action_hook_v2(tool_name: str, tool_input: dict, second_approver: str | None = None) -> bool:
    # TODO: reuse the delete_database and refund-over-limit rules, and add: block "delete_account"
    # unless second_approver is provided (not None)
    pass

print(destructive_action_hook_v2("delete_account", {}, second_approver=None))
print(destructive_action_hook_v2("delete_account", {}, second_approver="mgr_42"))

# Expected output: delete_account blocked without a second approver, allowed with one; existing rules still work.

In [ ]:
# STRETCH GOAL: log every blocked action with a reason to a list for later audit review.
audit_log = []
def destructive_action_hook_v3(tool_name, tool_input, second_approver=None):
    pass
destructive_action_hook_v3("delete_account", {}, second_approver=None)
print(audit_log)

# Expected output: An audit log entry recording the blocked delete_account attempt and the reason.

### Exercise 4: Secrets and authorization hardening

**Objective**: Fix a hard-coded secret and add a missing server-side authorization check.

**Produce**: A corrected version of a small vulnerable snippet.

In [ ]:
# VULNERABLE original:
def vulnerable_call_billing_api():
    api_key = "sk-hardcoded-12345"  # BUG: hard-coded secret
    return f"calling billing API with {api_key}"

def vulnerable_process_refund(user_role_from_client_request: str, amount: float):
    # BUG: trusts a role value that came directly from client-supplied input
    if user_role_from_client_request == "admin":
        return f"refunded {amount}"
    return "denied"

# TODO: fix both functions - use get_secret() for the API key, and use a trusted server-side
# lookup (simulate with a dict) instead of trusting client-supplied role for process_refund
def fixed_call_billing_api():
    pass

def fixed_process_refund(user_id: str, amount: float):
    pass

print(fixed_process_refund("u1", 100))

# Expected output: fixed_call_billing_api uses get_secret(); fixed_process_refund looks up the role server-side rather than trusting client input.

In [ ]:
# STRETCH GOAL: add rate limiting simulation - track refund attempts per user_id in a dict and
# block if more than 3 attempts occur within a simulated window.
attempt_counts = {}
def rate_limited_refund(user_id: str, amount: float):
    pass

# Expected output: Refund attempts beyond a threshold per user are blocked, simulating basic rate limiting.

## Self-Assessment: Security and Safety

Answer each question, then check the Answers cell below.


**Q1.** What is the difference between prompt injection and a jailbreak attempt?

**Q2.** Why is 'secure by design / least privilege' preferred over granting broad access and restricting later?

**Q3.** Why are hooks more reliable than prompt instructions for preventing destructive actions?

**Q4.** Where should secrets like API keys be stored, and where should they never be stored?

**Q5.** Why must authorization checks happen server-side rather than trusting a role/permission value from client input?

## Answers


**A1.** Prompt injection smuggles instructions via untrusted content the model processes; a jailbreak attempts to directly bypass the model's own safety training through crafted prompts.

**A2.** Because it minimizes the damage possible if a component is compromised or misused, rather than relying on remembering to restrict access later.

**A3.** Because hooks provide deterministic, code-enforced interception, while prompt instructions are only probabilistically followed by the model.

**A4.** In environment variables or a dedicated secrets manager, expanded at runtime; never hard-coded in source code or committed to version control.

**A5.** Because client-supplied input can be manipulated by an attacker; only server-side, trusted data sources can reliably determine actual permissions.

---
**Next**: [Domain 8: Tools and MCPs](./08-tools-and-mcps.ipynb)
**Back**: [README](./README.md)